# Detector Inference

Frozen inference of 4 detectors on DF40 dataset.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install torch torchvision pillow pandas tqdm pyyaml -q

import os
import sys
import json
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from datetime import datetime

# ── Chemins ───────────────────────────────────────────────────────────────────
PROJECT_ROOT  = '/content/drive/MyDrive/Memoire_Deepfakes'
BENCHMARK_SRC = f'{PROJECT_ROOT}/benchmark_src'
WEIGHTS_DIR   = f'{PROJECT_ROOT}/weights/pretrained'
SPLITS_DIR    = f'{PROJECT_ROOT}/data/splits'
RESULTS_DIR   = f'{PROJECT_ROOT}/data/results'

# Création du dossier results/ si inexistant
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Fichiers poids attendus ───────────────────────────────────────────────────
WEIGHT_FILES = {
    'XceptionNet': 'xception_best.pth',
    'UCF'        : 'ucf_best.pth',
    'Meso4'      : 'meso4_best.pth',
    'F3Net'      : 'f3net_best.pth',
}

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

print('=' * 65)
print('NOTEBOOK 04 — INFÉRENCE COMPLÈTE')
print('=' * 65)
print(f'  Date/Heure   : {datetime.now().strftime("%d/%m/%Y %H:%M")}')
print(f'  Timestamp    : {TIMESTAMP}')
print(f'  Device       : {DEVICE}')
print(f'  PyTorch      : {torch.__version__}')
print(f'  CUDA dispo   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU          : {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  VRAM         : {mem:.1f} GB')
print(f'  Results dir  : {RESULTS_DIR}')
print('=' * 65)

# Avertissement si pas de GPU
if not torch.cuda.is_available():
    print()
    print('  ⚠️  ATTENTION : GPU non détecté !')
    print('  → Exécution sur CPU sera très lente (plusieurs heures)')
    print('  → RECOMMANDÉ : Runtime → Modifier le type de matériel → GPU T4')
    print('  → Puis relancer ce notebook depuis le début')
else:
    print()
    print('  ✅ GPU disponible — inférence activée')


## DeepfakeBench Import

In [ ]:
print('=' * 65)
print('IMPORT ARCHITECTURES ET CHARGEMENT POIDS')
print('=' * 65)

import importlib
import importlib.util
import types
import yaml
import inspect
import traceback
import torch.nn as nn

TRAINING_DIR   = f'{BENCHMARK_SRC}/training'
detectors_path = f'{TRAINING_DIR}/detectors'
networks_path  = f'{TRAINING_DIR}/networks'
loss_path      = f'{TRAINING_DIR}/loss'
metrics_path   = f'{BENCHMARK_SRC}/metrics'

# ══════════════════════════════════════════════════════════════
# BLOC A — REGISTRES UNIVERSELS
# ══════════════════════════════════════════════════════════════

class _FakeRegistry:
    def __init__(self, name=''):
        self.name = name
        self._registry = {}

    def register_module(self, name=None, force=False, module=None, **kwargs):
        def decorator(cls):
            key = name if name else cls.__name__
            self._registry[key] = cls
            return cls
        if module is not None:
            return decorator(module)
        return decorator

    def build(self, cfg, *args, **kwargs):
        if isinstance(cfg, dict):
            cls = self._registry.get(cfg.get('type'))
            if cls:
                return cls(**{k: v for k, v in cfg.items() if k != 'type'})
        return None

    def __contains__(self, key): return key in self._registry
    def __getitem__(self, key):  return self._registry[key]
    def get(self, key, default=None): return self._registry.get(key, default)

DETECTOR_REG = _FakeRegistry('detector')
BACKBONE_REG = _FakeRegistry('backbone')
LOSS_REG     = _FakeRegistry('loss')
METRIC_REG   = _FakeRegistry('metric')
LOSSFUNC_REG = _FakeRegistry('lossfunc')

class _StubLoss(nn.Module):
    def __init__(self, *args, **kwargs): super().__init__()
    def forward(self, *args, **kwargs): return torch.tensor(0.0)

for _loss_key in ['cross_entropy', 'CrossEntropyLoss',
                   'bce', 'BCELoss', 'bce_with_logits',
                   'BCEWithLogitsLoss', 'am_softmax',
                   'contrastive', 'contrastive_regularization',
                   'focal', 'l1', 'l1loss', 'L1Loss',
                   'mse', 'MSELoss', 'rec_loss']:
    LOSSFUNC_REG._registry[_loss_key] = _StubLoss

# ══════════════════════════════════════════════════════════════
# BLOC B — FONCTION UTILITAIRE : création de stub
# ══════════════════════════════════════════════════════════════

def make_stub(full_name, package=None, path=None, **attrs):
    stub = types.ModuleType(full_name)
    stub.__package__ = package or full_name.rsplit('.', 1)[0]
    if path:
        stub.__path__ = [path]
        stub.__file__ = f'{path}/__init__.py'
    for k, v in attrs.items():
        setattr(stub, k, v)
    sys.modules[full_name] = stub
    return stub

# ══════════════════════════════════════════════════════════════
# BLOC C — PAQUETS PRINCIPAUX AVEC ATTRIBUTS COMPLETS
# ══════════════════════════════════════════════════════════════

detectors_pkg = make_stub(
    'detectors', package='detectors', path=detectors_path,
    DETECTOR=DETECTOR_REG
)
networks_pkg = make_stub(
    'networks', package='networks', path=networks_path,
    BACKBONE=BACKBONE_REG
)
loss_pkg = make_stub(
    'loss', package='loss', path=loss_path,
    LOSS=LOSS_REG,
    LOSSFUNC=LOSSFUNC_REG
)
metrics_pkg = make_stub(
    'metrics', package='metrics', path=metrics_path,
    METRIC=METRIC_REG,
    BACKBONE=BACKBONE_REG
)

# ══════════════════════════════════════════════════════════════
# BLOC D — STUBS SOUS-MODULES AVEC ATTRIBUTS PRÉCIS
# ══════════════════════════════════════════════════════════════

det_utils = make_stub('detectors.utils', package='detectors')
setattr(detectors_pkg, 'utils', det_utils)
det_base = make_stub('detectors.base_detector', package='detectors')
setattr(detectors_pkg, 'base_detector', det_base)

metrics_registry = make_stub(
    'metrics.registry', package='metrics',
    BACKBONE=BACKBONE_REG, DETECTOR=DETECTOR_REG,
    LOSS=LOSS_REG, LOSSFUNC=LOSSFUNC_REG, METRIC=METRIC_REG,
)
setattr(metrics_pkg, 'registry', metrics_registry)

def _dummy_metrics(*args, **kwargs): return {}
metrics_base = make_stub(
    'metrics.base_metrics_class', package='metrics',
    calculate_metrics_for_train=_dummy_metrics,
    calculate_metrics_for_test=_dummy_metrics,
    BaseMetrics=type('BaseMetrics', (), {
        'calculate_metrics_for_train': staticmethod(_dummy_metrics),
        'calculate_metrics_for_test' : staticmethod(_dummy_metrics),
    }),
)
setattr(metrics_pkg, 'base_metrics_class', metrics_base)
setattr(metrics_pkg, 'utils', make_stub('metrics.utils', package='metrics'))

for sub in ['resnet34', 'resnet50', 'resnet101', 'resnet152',
            'densenet', 'inception', 'vit', 'swin',
            'efficientnet', 'hrnet', 'efficientnetb4']:
    s = make_stub(f'networks.{sub}', package='networks')
    setattr(networks_pkg, sub, s)

for sub in ['abstract_loss_func', 'am_softmax', 'bce_loss',
            'cross_entropy_loss', 'contrastive_regularization',
            'consistency_loss', 'patch_consistency_loss',
            'region_independent_loss', 'supercontrast_loss',
            'vgg_loss', 'capsule_loss', 'js_loss', 'id_loss', 'l1_loss']:
    s = make_stub(f'loss.{sub}', package='loss',
                  LOSSFUNC=LOSSFUNC_REG, LOSS=LOSS_REG)
    setattr(loss_pkg, sub, s)

print('  ✅ Registres DETECTOR / BACKBONE / LOSS / LOSSFUNC / METRIC injectés')

# ══════════════════════════════════════════════════════════════
# BLOC E — CHARGEMENT RÉEL DES MODULES NETWORKS
# ══════════════════════════════════════════════════════════════

def load_module_from_file(full_name, filepath, package):
    try:
        spec = importlib.util.spec_from_file_location(full_name, filepath)
        mod  = importlib.util.module_from_spec(spec)
        mod.__package__ = package
        sys.modules[full_name] = mod
        spec.loader.exec_module(mod)
        return mod, None
    except Exception as e:
        stub = sys.modules.get(full_name) or make_stub(full_name, package=package)
        sys.modules[full_name] = stub
        return stub, str(e)

loaded_networks, failed_networks = [], []
if os.path.isdir(networks_path):
    for fname in sorted(os.listdir(networks_path)):
        if not fname.endswith('.py') or fname == '__init__.py':
            continue
        mod_name  = fname[:-3]
        full_name = f'networks.{mod_name}'
        fpath     = os.path.join(networks_path, fname)
        mod, err  = load_module_from_file(full_name, fpath, 'networks')
        setattr(networks_pkg, mod_name, mod)
        if err: failed_networks.append(f'{mod_name} ({err[:50]})')
        else:   loaded_networks.append(mod_name)

print(f'  ✅ networks/ réels chargés : {loaded_networks}')
if failed_networks:
    print(f'  ⚠️  networks/ en stub (non bloquant) : {failed_networks}')

base_det_path = f'{detectors_path}/base_detector.py'
if os.path.isfile(base_det_path):
    mod, err = load_module_from_file('detectors.base_detector', base_det_path, 'detectors')
    setattr(detectors_pkg, 'base_detector', mod)
    if err: print(f'  ⚠️  base_detector en stub : {err[:80]}')
    else:   print(f'  ✅ detectors.base_detector chargé depuis le disque')

# ══════════════════════════════════════════════════════════════
# BLOC E2 — ENREGISTREMENT MANUEL DES BACKBONES
# ══════════════════════════════════════════════════════════════

def register_backbone(module_name, class_names, keys):
    mod = sys.modules.get(f'networks.{module_name}')
    if mod is None:
        print(f'  ⚠️  Module networks.{module_name} absent — backbone non enregistré')
        return
    for cls_name in class_names:
        cls = getattr(mod, cls_name, None)
        if cls is not None:
            for key in keys:
                BACKBONE_REG._registry[key] = cls
            print(f'  ✅ BACKBONE{keys} → {cls_name} (depuis networks.{module_name})')
            return
    print(f'  ⚠️  Aucune des classes {class_names} trouvée dans networks.{module_name}')

register_backbone('xception', ['Xception', 'XceptionNet', 'XceptionModel'],
                  ['xception', 'Xception'])
register_backbone('mesonet',  ['Meso4', 'MesoNet', 'MesoInception', 'MesoInception4'],
                  ['meso4', 'mesonet', 'MesoNet', 'Meso4'])
print(f'  Clés BACKBONE_REG : {list(BACKBONE_REG._registry.keys())}')

# ══════════════════════════════════════════════════════════════
# BLOC F — sys.path
# ══════════════════════════════════════════════════════════════

for p in [TRAINING_DIR, BENCHMARK_SRC]:
    if p not in sys.path:
        sys.path.insert(0, p)

print()

# ══════════════════════════════════════════════════════════════
# BLOC G — CONFIGURATION DES 4 MODÈLES
# ══════════════════════════════════════════════════════════════

CONFIG_DIR = f'{TRAINING_DIR}/config/detector'

def load_yaml_config(yaml_path):
    with open(yaml_path, 'r') as f:
        cfg = yaml.safe_load(f)
    return cfg if cfg else {}

DEFAULTS = {
    'backbone_name'   : 'xception',
    'backbone_config' : None,
    'pretrained'      : False,
    'num_classes'     : 2,
    'compression'     : 'c23',
    'train_batchSize' : 32,
    'test_batchSize'  : 32,
    'workers'         : 4,
    'lr'              : 0.0002,
    'beta1'           : 0.5,
    'datapath'        : '',
    'normalize'       : {'mean': [0.5, 0.5, 0.5], 'std': [0.5, 0.5, 0.5]},
    'image_size'      : 256,
    'with_landmark'   : False,
    'with_mask'       : False,
    'lnum'            : 0,
    'device'          : 'cuda',
    'logdir'          : '',
    'manualSeed'      : 42,
    'label_dict'      : {'FAKE': 1, 'REAL': 0},
    'clip_size'       : 8,
    'frame_num'       : {'train': 1, 'test': 1, 'val': 1},
    'data_manner'     : 'image',
}

MODEL_SPECIFIC = {
    'Meso4'      : {'backbone_name': 'meso4'},
    'XceptionNet': {'backbone_name': 'xception'},
    'UCF'        : {'backbone_name': 'xception',
                    'backbone_config': {'num_classes': 2, 'inc': 3, 'dropout': False},
                    'head_type'      : 'SVM',
                    'num_clusters'   : 32,
                    'svm_c'          : 1.0},
    'F3Net'      : {'backbone_name': 'xception',
                    'FAD_Head_size'  : 768,
                    'LFS_window_size': 10,
                    'LFS_M'         : 6,
                    'LFS_stride'    : 16},
}

MODEL_CONFIGS = [
    {
        'name'       : 'Meso4',
        'filepath'   : f'{detectors_path}/meso4_detector.py',
        'class_name' : 'Meso4Detector',
        'weight_file': os.path.join(WEIGHTS_DIR, WEIGHT_FILES['Meso4']),
        'config_file': f'{CONFIG_DIR}/meso4.yaml',
    },
    {
        'name'       : 'XceptionNet',
        'filepath'   : f'{detectors_path}/xception_detector.py',
        'class_name' : 'XceptionDetector',
        'weight_file': os.path.join(WEIGHTS_DIR, WEIGHT_FILES['XceptionNet']),
        'config_file': f'{CONFIG_DIR}/xception.yaml',
    },
    {
        'name'       : 'UCF',
        'filepath'   : f'{detectors_path}/ucf_detector.py',
        'class_name' : 'UCFDetector',
        'weight_file': os.path.join(WEIGHTS_DIR, WEIGHT_FILES['UCF']),
        'config_file': f'{CONFIG_DIR}/ucf.yaml',
    },
    {
        'name'       : 'F3Net',
        'filepath'   : f'{detectors_path}/f3net_detector.py',
        'class_name' : 'F3netDetector',
        'weight_file': os.path.join(WEIGHTS_DIR, WEIGHT_FILES['F3Net']),
        'config_file': f'{CONFIG_DIR}/f3net.yaml',
    },
]

# ══════════════════════════════════════════════════════════════
# BLOC H — MONKEY-PATCH torch.load + BOUCLE PRINCIPALE
# ══════════════════════════════════════════════════════════════

class _DummyStateDict:
    """Répond à toute clé avec un tenseur factice.
    load_state_dict() écrasera tout avec les vrais poids _best.pth."""
    def __getitem__(self, key):       return torch.zeros(64, 3, 3, 3)
    def __contains__(self, key):      return True
    def get(self, key, default=None): return torch.zeros(64, 3, 3, 3)
    def items(self):  return {}.items()
    def keys(self):   return {}.keys()
    def values(self): return {}.values()

_original_torch_load = torch.load

def _safe_torch_load(f, *args, **kwargs):
    if isinstance(f, (bool, type(None))):
        return _DummyStateDict()
    if isinstance(f, str) and not os.path.isfile(f):
        return _DummyStateDict()
    return _original_torch_load(f, *args, **kwargs)

torch.load = _safe_torch_load

loaded_models = {}

for cfg in MODEL_CONFIGS:
    name = cfg['name']
    print(f'  [{name}]')

    try:
        model_config = load_yaml_config(cfg['config_file'])
        print(f'    ✅ Config YAML : {os.path.basename(cfg["config_file"])}')
    except Exception as e:
        print(f'    ⚠️  YAML introuvable ({e}) — config vide utilisée')
        model_config = {}

    module_name = f'detectors.{os.path.basename(cfg["filepath"])[:-3]}'
    mod, err = load_module_from_file(module_name, cfg['filepath'], 'detectors')
    setattr(detectors_pkg, os.path.basename(cfg['filepath'])[:-3], mod)

    if err:
        print(f'    ⚠️  ÉCHEC import fichier : {err}')
        continue
    print(f'    ✅ Fichier chargé : {os.path.basename(cfg["filepath"])}')

    ModelClass = getattr(mod, cfg['class_name'], None)
    if ModelClass is None:
        classes = [n for n, obj in inspect.getmembers(mod, inspect.isclass)]
        print(f'    ⚠️  Classe "{cfg["class_name"]}" introuvable. Disponibles : {classes}')
        continue
    print(f'    ✅ Classe : {cfg["class_name"]}')

    # ── Patch ciblé F3Net uniquement ──────────────────────────
    if name == 'F3Net':
        def _safe_f3net_build_backbone(self, config):
            backbone_class = BACKBONE_REG._registry.get('xception')
            if backbone_class is None:
                raise RuntimeError('Backbone xception absent de BACKBONE_REG')
            xception_config = {
                'num_classes': config.get('num_classes', 2),
                'mode'       : config.get('mode', 'original'),
                'inc'        : 12,
                'dropout'    : config.get('dropout', False),
            }
            return backbone_class(xception_config)
        ModelClass.build_backbone = _safe_f3net_build_backbone
        print(f'    ✅ build_backbone patché — skip xception-b5690688.pth')

    merged_config = {**DEFAULTS, **MODEL_SPECIFIC.get(name, {}), **model_config}
    merged_config['pretrained'] = False

    model = None
    try:
        model = ModelClass(merged_config)
        print(f'    ✅ Instanciation réussie')
    except Exception as e:
        print(f'    ⚠️  Échec instanciation : {e}')
        traceback.print_exc()
        continue

    try:
        ckpt = _original_torch_load(cfg['weight_file'], map_location='cpu')

        if isinstance(ckpt, dict):
            for key in ['state_dict', 'model', 'net', 'params']:
                if key in ckpt:
                    state_dict = ckpt[key]
                    break
            else:
                state_dict = ckpt
        else:
            state_dict = ckpt

        state_dict = {
            (k[len('module.'):] if k.startswith('module.') else k): v
            for k, v in state_dict.items()
        }

        # Suppression préfixe DataParallel
        state_dict = {
            (k[len('module.'):] if k.startswith('module.') else k): v
            for k, v in state_dict.items()
        }

        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        n_m, n_u = len(missing), len(unexpected)

        if n_m == 0 and n_u == 0:
            print(f'    ✅ Poids : 0 missing, 0 unexpected')
        elif n_m <= 15 and n_u <= 15:
            print(f'    ✅ Poids : missing={n_m}, unexpected={n_u} (acceptable)')
            if missing:    print(f'       missing    : {missing[:3]}')
            if unexpected: print(f'       unexpected : {unexpected[:3]}')
        else:
            print(f'    ⚠️  Chargement partiel : missing={n_m}, unexpected={n_u}')

        model.eval()
        loaded_models[name] = model
        print(f'    ✅ eval() — modèle prêt')

    except Exception as e:
        print(f'    ⚠️  ÉCHEC chargement poids : {e}')
        traceback.print_exc()

torch.load = _original_torch_load
print()
print(f'  {len(loaded_models)}/4 modèles chargés : {list(loaded_models.keys())}')
assert len(loaded_models) == 4, (
    f'ARRÊT — {4 - len(loaded_models)} modèle(s) manquant(s). '
    f'Relancer cette cellule ou vérifier les fichiers .pth et .py.'
)
print('  ✅ 4/4 modèles prêts pour l\'inférence')


## Load Manifests

In [ ]:
print('=' * 65)
print('CHARGEMENT DES MANIFESTES')
print('=' * 65)

# ⚠️ val_manifest.csv EXCLU — coffre-fort inviolable
SAFE_SPLITS = ['train', 'test']
EXPECTED_ROWS = {'train': 1402, 'test': 2807}

manifests = {}
for split_name in SAFE_SPLITS:
    csv_path = f'{SPLITS_DIR}/{split_name}_manifest.csv'
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(
            f'Manifeste introuvable : {csv_path}\n'
            f'→ Relancer le notebook 03 pour régénérer les splits'
        )
    df = pd.read_csv(csv_path)
    manifests[split_name] = df

    n_real = int((df['label'] == 0).sum())
    n_fake = int((df['label'] == 1).sum())
    print(f'\n  {split_name}_manifest.csv :')
    print(f'    Lignes     : {len(df):,} (attendu : {EXPECTED_ROWS[split_name]:,})')
    print(f'    Colonnes   : {list(df.columns)}')
    print(f'    REAL (0)   : {n_real:,}')
    print(f'    FAKE (1)   : {n_fake:,}')
    print(f'    Ratio      : {n_real / max(n_fake, 1):.4f}')
    if 'method' in df.columns:
        method_dist = df[df['label'] == 1]['method'].value_counts().to_dict()
        print(f'    Méthodes   : {method_dist}')

    # Vérification : aucune ligne val dans les manifestes train/test
    if 'split' in df.columns:
        bad = df[df['split'] == 'val']
        if len(bad) > 0:
            raise ValueError(
                f'⚠️  {len(bad)} lignes val détectées dans {split_name}_manifest — ARRÊT'
            )

    # Vérification comptage
    if len(df) != EXPECTED_ROWS[split_name]:
        print(f'    ⚠️  ATTENTION : {len(df)} lignes ≠ {EXPECTED_ROWS[split_name]} attendues')

total_images = sum(len(v) for v in manifests.values())
print()
print(f'  Total images à inférer : {total_images:,}')
print(f'  (train={len(manifests["train"]):,} + test={len(manifests["test"]):,})')
print()
print('  ✅ Manifestes chargés — val_manifest.csv NON chargé (coffre-fort)')


## Preprocessing & Dataset

In [ ]:
import torchvision.transforms as T
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print('=' * 65)
print('PREPROCESSING ET DATASET')
print('=' * 65)

# ── Transform standard ───────────────────────────────────────────────────────
# Identique aux DEFAULTS configurés dans notebook 03b
TRANSFORM = T.Compose([
    T.Resize((256, 256),
             interpolation=T.InterpolationMode.BILINEAR,
             antialias=True),           # défensif — images déjà 256×256
    T.ToTensor(),                        # PIL [0,255] → FloatTensor [0,1]
    T.Normalize(mean=[0.5, 0.5, 0.5],   # [0,1] → [-1, 1]
                std =[0.5, 0.5, 0.5]),
])

# ── Dataset ──────────────────────────────────────────────────────────────────
class FaceDataset(Dataset):
    """
    Dataset PyTorch pour les visages pré-croppés 256×256 de DF40.

    Returns:
        (tensor, filepath) :  tensor de shape (3, 256, 256) normalisé [-1,1]
                               filepath : str — chemin absolu de l'image
    En cas d'image corrompue, retourne un tensor de zéros et logue un warning.
    """
    def __init__(self, df: pd.DataFrame, transform):
        self.records   = df[['filepath']].reset_index(drop=True)
        self.transform = transform
        self.n_errors  = 0

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        filepath = str(self.records.iloc[idx]['filepath'])
        try:
            img    = Image.open(filepath).convert('RGB')
            tensor = self.transform(img)
        except Exception:
            # Image corrompue ou inaccessible : tensor noir + comptage
            self.n_errors += 1
            tensor = torch.zeros(3, 256, 256)
        return tensor, filepath


def make_dataloader(df: pd.DataFrame, batch_size: int) -> DataLoader:
    """Crée un DataLoader non-mélangé (shuffle=False) depuis un manifeste.

    num_workers=0 : évite les problèmes de montage Drive dans les workers.
    drop_last=False : conserve le dernier batch incomplet.
    """
    dataset = FaceDataset(df, TRANSFORM)
    return DataLoader(
        dataset,
        batch_size  = batch_size,
        shuffle     = False,
        num_workers = 0,
        pin_memory  = torch.cuda.is_available(),
        drop_last   = False,
    )

# ── Validation sur la première image de train ─────────────────────────────────
first_row    = manifests['train'].iloc[0]
test_img     = Image.open(first_row['filepath']).convert('RGB')
test_tensor  = TRANSFORM(test_img)

print(f'  Image test : {os.path.basename(first_row["filepath"])}')
print(f'  Taille PIL : {test_img.size}  →  Tensor : {tuple(test_tensor.shape)}')
print(f'  Min / Max  : [{test_tensor.min():.4f}, {test_tensor.max():.4f}]')
print(f'  Mean       : {test_tensor.mean():.4f}')

assert test_tensor.shape == (3, 256, 256), \
    f'Shape attendu (3, 256, 256), obtenu {tuple(test_tensor.shape)}'
assert test_tensor.min() >= -1.1 and test_tensor.max() <= 1.1, \
    f'Valeurs hors [-1, 1] : [{test_tensor.min():.4f}, {test_tensor.max():.4f}]'

print()
print('  ✅ Pipeline preprocessing validée')
print(f'  ✅ FaceDataset et make_dataloader définis')


## Inference Function

In [ ]:
from tqdm.auto import tqdm

print('=' * 65)
print('FONCTION D\'INFÉRENCE GÉNÉRIQUE')
print('=' * 65)

# ── Batch size par défaut ─────────────────────────────────────────────────────
BATCH_SIZE_DEFAULT = 32   # Meso4, XceptionNet, F3Net
BATCH_SIZE_UCF     = 2    # UCF — chunk(2, dim=0) dans forward()


def _extract_prob_fake(output) -> torch.Tensor:
    """
    Extrait un vecteur de probabilités P(FAKE) depuis la sortie brute du modèle.

    Cas couverts :
      - output est un dict {'cls': logits, ...}  → clé 'cls' (XceptionNet, UCF, F3Net)
      - output est un tenseur (N, 2)              → softmax[:, 1]
      - output est un tenseur (N, 1)              → sigmoid[:, 0]
      - output est un tuple/list                  → premier élément

    Returns:
        Tenseur 1D float, longueur = taille du batch
    """
    # Extraction des logits depuis un éventuel dict
    if isinstance(output, dict):
        logits = output.get('cls',
                 output.get('logits',
                 output.get('pred',
                 list(output.values())[0])))
    else:
        logits = output

    # Dépliage liste/tuple
    if isinstance(logits, (tuple, list)):
        logits = logits[0]

    # Garantir 2D
    if logits.ndim == 1:
        logits = logits.unsqueeze(0)

    # Conversion logits → probabilité P(FAKE)
    n_classes = logits.shape[-1]
    if n_classes == 2:
        return F.softmax(logits, dim=1)[:, 1]   # classe 1 = FAKE
    elif n_classes == 1:
        return torch.sigmoid(logits).squeeze(1)
    else:
        # Cas inattendu — sigmoid sur la dernière sortie
        return torch.sigmoid(logits[:, -1])


def _forward_one_batch(
    model,
    imgs   : torch.Tensor,
    device : torch.device,
) -> torch.Tensor:
    """
    Exécute un forward pass sur un batch d'images.
    Essaie d'abord l'interface data_dict, puis le tenseur brut en fallback.

    Args:
        imgs   : (N, 3, 256, 256) — déjà sur device
    Returns:
        tenseur 1D (N,) de probabilités P(FAKE)
    """
    n_batch = imgs.shape[0]
    with torch.no_grad():
        try:
            data_dict = {
                'image'   : imgs,
                'label'   : torch.zeros(n_batch, dtype=torch.long).to(device),
                'mask'    : None,
                'landmark': None,
            }
            output = model(data_dict)
        except (TypeError, KeyError, AttributeError):
            # Fallback interface tenseur brut (Meso4)
            output = model(imgs)
    return _extract_prob_fake(output)


def run_inference(
    model_name  : str,
    df_manifest : pd.DataFrame,
    device      : torch.device,
    split_label : str = '',
) -> dict:
    """
    Exécute l'inférence complète d'un modèle sur un manifeste.

    Args:
        model_name  : clé dans loaded_models ('Meso4', 'XceptionNet', 'UCF', 'F3Net')
        df_manifest : DataFrame du split (train ou test, JAMAIS val)
        device      : CPU ou CUDA
        split_label : label d'affichage pour tqdm

    Returns:
        dict {filepath (str) : prob_fake (float)}

    Note UCF :
        Batch size forcé à 2. Si le dernier batch contient 1 seule image,
        elle est dupliquée avant le forward pass et la prédiction dupliquée
        est supprimée du résultat.
    """
    assert model_name in loaded_models, \
        f'Modèle {model_name!r} absent de loaded_models — vérifier cellule 2'

    model = loaded_models[model_name]
    model = model.to(device)
    model.eval()

    batch_size = BATCH_SIZE_UCF if model_name == 'UCF' else BATCH_SIZE_DEFAULT
    loader     = make_dataloader(df_manifest, batch_size=batch_size)

    filepath_to_prob : dict = {}
    error_list       : list = []

    desc = f'  {model_name:<12s} [{split_label}]' if split_label else f'  {model_name}'

    for imgs, filepaths in tqdm(loader, desc=desc, unit='batch'):
        actual_bs     = imgs.shape[0]
        needs_truncate = False

        # ── Padding obligatoire UCF (batch = 1 → dupliquer) ──────────────
        if model_name == 'UCF' and actual_bs < 2:
            imgs           = torch.cat([imgs, imgs], dim=0)  # (1,C,H,W)→(2,C,H,W)
            needs_truncate = True

        imgs = imgs.to(device)

        try:
            probs_tensor = _forward_one_batch(model, imgs, device)  # (N,)
            probs_list   = probs_tensor.cpu().tolist()

            # Suppression de la prédiction dupliquée pour UCF
            if needs_truncate:
                probs_list = probs_list[:actual_bs]
                filepaths  = list(filepaths)[:actual_bs]

            for fp, prob in zip(filepaths, probs_list):
                filepath_to_prob[str(fp)] = float(prob)

        except Exception as exc:
            # En cas d'erreur sur un batch : NaN + log
            for fp in list(filepaths)[:actual_bs]:
                filepath_to_prob[str(fp)] = float('nan')
                error_list.append((str(fp), str(exc)))

    if error_list:
        print(f'\n    ⚠️  {len(error_list)} erreur(s) de forward pass :')
        for fp, err in error_list[:3]:
            print(f'       {os.path.basename(fp)} : {err[:60]}')

    return filepath_to_prob


def _save_intermediate(probs_dict: dict, model_name: str) -> str:
    """
    Sauvegarde intermédiaire dans data/results/tmp_{model_name}_probs.csv.
    Protection contre les déconnexions Colab entre les cellules d'inférence.
    """
    col   = f'P_{model_name}'
    df    = pd.DataFrame([
        {'filepath': k, col: v} for k, v in probs_dict.items()
    ])
    path  = f'{RESULTS_DIR}/tmp_{model_name.lower()}_probs.csv'
    df.to_csv(path, index=False)
    return path


def _load_intermediate(model_name: str) -> dict:
    """
    Recharge un fichier intermédiaire si disponible (reprise après déconnexion).
    Retourne {} si le fichier n'existe pas.
    """
    col  = f'P_{model_name}'
    path = f'{RESULTS_DIR}/tmp_{model_name.lower()}_probs.csv'
    if os.path.isfile(path):
        df = pd.read_csv(path)
        if 'filepath' in df.columns and col in df.columns:
            return dict(zip(df['filepath'].tolist(), df[col].tolist()))
    return {}


print('  ✅ Fonctions définies :')
print('     _extract_prob_fake      — extraction P(FAKE) depuis output')
print('     _forward_one_batch      — forward pass avec fallback data_dict/tenseur')
print('     run_inference           — inférence complète sur un split')
print('     _save_intermediate      — sauvegarde CSV partielle (protection Colab)')
print('     _load_intermediate      — reprise si déconnexion')
print()
print('  Paramètres batch :')
print(f'     BATCH_SIZE_DEFAULT (Meso4, XceptionNet, F3Net) : {BATCH_SIZE_DEFAULT}')
print(f'     BATCH_SIZE_UCF                                 : {BATCH_SIZE_UCF}')


## Meso4 Inference

In [ ]:
print('=' * 65)
print('INFÉRENCE MESO4')
print('=' * 65)

MODEL_NAME = 'Meso4'

# Tentative de reprise si fichier intermédiaire disponible
probs_meso4 = _load_intermediate(MODEL_NAME)
if probs_meso4:
    print(f'  ℹ️  Reprise détectée : {len(probs_meso4):,} prédictions chargées depuis tmp CSV')
else:
    probs_meso4 = {}
    for split_name in ['train', 'test']:
        print(f'  Split : {split_name}')
        partial = run_inference(
            model_name  = MODEL_NAME,
            df_manifest = manifests[split_name],
            device      = DEVICE,
            split_label = split_name,
        )
        n_nan = sum(1 for v in partial.values() if v != v)  # float nan != nan
        print(f'    → {len(partial):,} prédictions | NaN : {n_nan}')
        probs_meso4.update(partial)

    tmp_path = _save_intermediate(probs_meso4, MODEL_NAME)
    print(f'  💾 Sauvegarde intermédiaire → {os.path.basename(tmp_path)}')

vals = [v for v in probs_meso4.values() if v == v]
print()
print(f'  ✅ Meso4 terminé : {len(probs_meso4):,} prédictions')
print(f'     Min={min(vals):.4f} | Max={max(vals):.4f} | Mean={sum(vals)/len(vals):.4f}')


## XceptionNet Inference

In [ ]:
print('=' * 65)
print('INFÉRENCE XCEPTIONNET')
print('=' * 65)

MODEL_NAME = 'XceptionNet'

probs_xception = _load_intermediate(MODEL_NAME)
if probs_xception:
    print(f'  ℹ️  Reprise détectée : {len(probs_xception):,} prédictions chargées depuis tmp CSV')
else:
    probs_xception = {}
    for split_name in ['train', 'test']:
        print(f'  Split : {split_name}')
        partial = run_inference(
            model_name  = MODEL_NAME,
            df_manifest = manifests[split_name],
            device      = DEVICE,
            split_label = split_name,
        )
        n_nan = sum(1 for v in partial.values() if v != v)
        print(f'    → {len(partial):,} prédictions | NaN : {n_nan}')
        probs_xception.update(partial)

    tmp_path = _save_intermediate(probs_xception, MODEL_NAME)
    print(f'  💾 Sauvegarde intermédiaire → {os.path.basename(tmp_path)}')

vals = [v for v in probs_xception.values() if v == v]
print()
print(f'  ✅ XceptionNet terminé : {len(probs_xception):,} prédictions')
print(f'     Min={min(vals):.4f} | Max={max(vals):.4f} | Mean={sum(vals)/len(vals):.4f}')


## UCF Inference

⚠️ batch_size=2 required for memory constraints.

In [ ]:
print('=' * 65)
print('INFÉRENCE UCF (batch_size=2)')
print('=' * 65)

MODEL_NAME = 'UCF'

probs_ucf = _load_intermediate(MODEL_NAME)
if probs_ucf:
    print(f'  ℹ️  Reprise détectée : {len(probs_ucf):,} prédictions chargées depuis tmp CSV')
else:
    probs_ucf = {}
    for split_name in ['train', 'test']:
        print(f'  Split : {split_name}')
        print(f'  (batch_size=2 → {len(manifests[split_name])//2 + len(manifests[split_name])%2} batches)')
        partial = run_inference(
            model_name  = MODEL_NAME,
            df_manifest = manifests[split_name],
            device      = DEVICE,
            split_label = split_name,
        )
        n_nan = sum(1 for v in partial.values() if v != v)
        print(f'    → {len(partial):,} prédictions | NaN : {n_nan}')
        probs_ucf.update(partial)

    tmp_path = _save_intermediate(probs_ucf, MODEL_NAME)
    print(f'  💾 Sauvegarde intermédiaire → {os.path.basename(tmp_path)}')

vals = [v for v in probs_ucf.values() if v == v]
print()
print(f'  ✅ UCF terminé : {len(probs_ucf):,} prédictions')
print(f'     Min={min(vals):.4f} | Max={max(vals):.4f} | Mean={sum(vals)/len(vals):.4f}')


## F3Net Inference

In [ ]:
print('=' * 65)
print('INFÉRENCE F3NET')
print('=' * 65)

MODEL_NAME = 'F3Net'

probs_f3net = _load_intermediate(MODEL_NAME)
if probs_f3net:
    print(f'  ℹ️  Reprise détectée : {len(probs_f3net):,} prédictions chargées depuis tmp CSV')
else:
    probs_f3net = {}
    for split_name in ['train', 'test']:
        print(f'  Split : {split_name}')
        partial = run_inference(
            model_name  = MODEL_NAME,
            df_manifest = manifests[split_name],
            device      = DEVICE,
            split_label = split_name,
        )
        n_nan = sum(1 for v in partial.values() if v != v)
        print(f'    → {len(partial):,} prédictions | NaN : {n_nan}')
        probs_f3net.update(partial)

    tmp_path = _save_intermediate(probs_f3net, MODEL_NAME)
    print(f'  💾 Sauvegarde intermédiaire → {os.path.basename(tmp_path)}')

vals = [v for v in probs_f3net.values() if v == v]
print()
print(f'  ✅ F3Net terminé : {len(probs_f3net):,} prédictions')
print(f'     Min={min(vals):.4f} | Max={max(vals):.4f} | Mean={sum(vals)/len(vals):.4f}')


## Assembly & Save

In [ ]:
print('=' * 65)
print('ASSEMBLAGE ET SAUVEGARDE DES CSVs FINAUX')
print('=' * 65)

# Vérification que les 4 dicts de probabilités sont disponibles
# (si une cellule a échoué, tentative de reprise depuis les tmp CSV)
_probs_registry = {
    'Meso4'      : {'var': 'probs_meso4',    'col': 'P_Meso4'},
    'XceptionNet': {'var': 'probs_xception',  'col': 'P_XceptionNet'},
    'UCF'        : {'var': 'probs_ucf',       'col': 'P_UCF'},
    'F3Net'      : {'var': 'probs_f3net',     'col': 'P_F3Net'},
}

# Récupération sécurisée des dicts (fallback tmp CSV si variable absente)
_all_probs = {}
for model_name, info in _probs_registry.items():
    var_dict = globals().get(info['var'], {})
    if not var_dict:
        print(f'  ⚠️  {info["var"]} vide — tentative de reprise depuis tmp CSV')
        var_dict = _load_intermediate(model_name)
        if var_dict:
            print(f'     ✅ {len(var_dict):,} prédictions rechargées')
        else:
            print(f'     ⚠️  Aucune donnée disponible pour {model_name}')
    _all_probs[model_name] = var_dict

print()

# Colonnes de sortie attendues (dans l'ordre spécifié dans le journal de suivi)
OUTPUT_COLS = ['filepath', 'label', 'method', 'split',
               'P_Meso4', 'P_XceptionNet', 'P_UCF', 'P_F3Net']

saved_paths = {}
for split_name in ['train', 'test']:
    df_split = manifests[split_name].copy()

    # Ajout des colonnes de probabilité
    for model_name, info in _probs_registry.items():
        col      = info['col']
        probs_d  = _all_probs[model_name]
        df_split[col] = df_split['filepath'].map(probs_d)

    # Sélection et réordonnancement des colonnes disponibles
    available_cols = [c for c in OUTPUT_COLS if c in df_split.columns]
    df_out         = df_split[available_cols].copy()

    # Diagnostic NaN
    prob_cols  = [c for c in available_cols if c.startswith('P_')]
    total_nan  = 0
    for col in prob_cols:
        n_nan = int(df_out[col].isna().sum())
        total_nan += n_nan
        if n_nan > 0:
            print(f'  ⚠️  {split_name} / {col} : {n_nan} NaN')

    # Sauvegarde
    out_path = f'{RESULTS_DIR}/{split_name}_probs.csv'
    df_out.to_csv(out_path, index=False)
    saved_paths[split_name] = out_path

    nan_flag = f'⚠️  {total_nan} NaN' if total_nan > 0 else '✅ 0 NaN'
    print(f'  ✅ {split_name}_probs.csv sauvegardé')
    print(f'     Lignes   : {len(df_out):,}')
    print(f'     Colonnes : {list(df_out.columns)}')
    print(f'     NaN      : {nan_flag}')
    print(f'     Chemin   : {out_path}')
    print()

# Nettoyage des fichiers temporaires intermédiaires
print('  Nettoyage fichiers intermédiaires :')
for model_name in _probs_registry:
    tmp_path = f'{RESULTS_DIR}/tmp_{model_name.lower()}_probs.csv'
    if os.path.isfile(tmp_path):
        os.remove(tmp_path)
        print(f'    🗑  Supprimé : {os.path.basename(tmp_path)}')

print()
print('  ✅ CSVs finaux sauvegardés dans data/results/')


## Verification

In [ ]:
print('=' * 65)
print('NOTEBOOK 04 — RÉSUMÉ FINAL')
print('=' * 65)

EXPECTED = {'train': 1402, 'test': 2807}
PROB_COLS = ['P_Meso4', 'P_XceptionNet', 'P_UCF', 'P_F3Net']

all_checks_ok = True
for split_name in ['train', 'test']:
    out_path = f'{RESULTS_DIR}/{split_name}_probs.csv'
    print(f'\n  {split_name}_probs.csv :')

    if not os.path.isfile(out_path):
        print(f'    ⚠️  ABSENT — relancer cellule 10')
        all_checks_ok = False
        continue

    df = pd.read_csv(out_path)
    exp_rows = EXPECTED[split_name]
    rows_ok  = len(df) == exp_rows
    print(f'    Lignes     : {len(df):,} / {exp_rows:,} attendues  {"✅" if rows_ok else "⚠️"}')

    for col in PROB_COLS:
        if col not in df.columns:
            print(f'    ⚠️  Colonne absente : {col}')
            all_checks_ok = False
            continue
        vals  = df[col].dropna()
        n_nan = int(df[col].isna().sum())
        in_range = (vals.min() >= 0.0) and (vals.max() <= 1.0)
        flag = '✅' if (n_nan == 0 and in_range) else '⚠️'
        print(f'    {flag} {col:<18s}: min={vals.min():.4f} max={vals.max():.4f} '
              f'mean={vals.mean():.4f} NaN={n_nan}')
        if n_nan > 0 or not in_range:
            all_checks_ok = False

    if not rows_ok:
        all_checks_ok = False

# Vérification que val_manifest n'a pas été touché
val_path = f'{SPLITS_DIR}/val_manifest.csv'
val_loaded_flag = os.path.isfile(f'{RESULTS_DIR}/val_probs.csv')
print()
print(f'  Coffre-fort val_manifest : {"⚠️  val_probs.csv détecté !" if val_loaded_flag else "✅ INTACT"}')

print()
print('=' * 65)
if all_checks_ok and not val_loaded_flag:
    print('  ✅ NOTEBOOK 04 TERMINÉ AVEC SUCCÈS')
    print()
    print('  Fichiers générés dans data/results/ :')
    print('    → train_probs.csv  (1 402 lignes × 8 colonnes)')
    print('    → test_probs.csv   (2 807 lignes × 8 colonnes)')
    print()
    print('  Prochaine étape : Notebook 05 — Scénarios ensemble A / B / C')
    print('    Scénario A : Vote majoritaire (seuil 0.5, tie-break FAKE)')
    print('    Scénario B : Moyenne pondérée (poids ∝ précision Test Set individuelle)')
    print('    Scénario C : Méta-Learner (régression logistique, train sur Train Set)')
else:
    print('  ⚠️  PROBLÈMES DÉTECTÉS — Corriger avant notebook 05')
    print()
    print('  Guide de débogage :')
    print('    [Lignes incorrectes]  → Vérifier les manifestes en cellule 3')
    print('    [NaN dans les proba]  → Image(s) corrompue(s) ou erreur forward pass')
    print('    [Colonne absente]     → Relancer la cellule d\'inférence concernée')
    print('    [UCF NaN excessifs]   → Vérifier batch_size=2 et le padding')
print('=' * 65)
